# Nightjar Music Generation - based on RAVE


In order to generate 'Nightjar Music', this notebook uses the previously trained RAVE-model

```text
epoch_2000000.ckpt.ts
```

- The name is given by the fact that RAVE creates checkpoints during training after a certain number of steps/epochs. This model was trained with 2000000 RAVE steps.
- The model was trained on the training data provided in `data/nj_training_data`. It included bird/nature sounds, the Nightjar song itself, as well as more songs by Cosmo Sheldrake for more musical inspiration. Data augmentation was applied: The model was trained using 20 seconds audio files; mono; 441000Htz. To increase the number of song snippets for training, offset of 0/5/10 seconds was used.

___

How is the music generated?

The basic idea is to interpolate between points of existing Nightjar snippets in the latent space. Along the line between those two points, we take multiple points. By decoding them from latent space back to real audio, we will generate 'new' music.

---

Data source:

```text
data/nj_music_generation_data/nightjar_song   # Nightjar song; snippets of the song will be encoded and then used as fixed points in latent space
data/nj_training_data/                        # bird audio and songs that were used for training
```

The additional songs by Cosmo Sheldrake were only used for RAVE training; they are therefore only implicitly used as an effect on the decoded latent points.

---

The goal is to make a transition from the original Nightjar song to the generated song. Our generated song will be around 3 minutes long. We decided that the first part will be bird/nature sounds, since this is how the original Nightjar ends (its last ~17seconds are only few bird calls). We will also end the song with bird sounds, to generate a loop for the Nightjar project. 

Our song will therefore look as follows:


```text
Part I  nature/bird sounds intro
Part II  nightjar-based music; wandering around latent space
Part III  back to nature/birds; outro
```

(!! add Part for transition from birds to nightjar song?)

```text
Audio → RAVE encoder → Latent walk → RAVE decoder → Audio
```
---

1. Load the trained RAVE TorchScript model.
2. Load the full Nightjar song plus prepared training snippets.
3. Encode all audio into the RAVE latent space.
4. Visualize the latent space with PCA.
5. Score Nightjar snippets by simple mood/intensity features.
6. Select bird anchors and music anchors.
7. Generate a new song with **anchor holds** and a fixed number of **generated intermediate latent points** between anchors.
8. Visualize the generated path, including the generated intermediate points.

The key idea is that the output does **not** jump directly from one anchor to another. For every transition between two anchors, the notebook creates `NUM_INTERMEDIATE_POINTS` latent points between them and decodes those points into audio.





## 0. Installation


In [ ]:
# pip install -r requirements.txt

## 1. Imports and configuration


In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa
import librosa.display
import soundfile as sf

from tqdm.auto import tqdm
from IPython.display import Audio, display, Markdown, HTML

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import torch
from matplotlib.animation import FuncAnimation

warnings.filterwarnings("ignore")


In [ ]:
# ============================================================
# Paths
# ============================================================
RAVE_MODEL_PATH = Path("epoch_2000000.ckpt.ts")

NJ_GENERATION_DATA_DIR = Path("data/nj_music_generation_data")
NIGHTJAR_SONG = NJ_GENERATION_DATA_DIR / "nightjar_song" / "nightjar.wav"
BIRDS_FOR_GENERATION_DIR = NJ_GENERATION_DATA_DIR / "birds"
SIMILAR_SONGS_FOR_GENERATION_DIR = NJ_GENERATION_DATA_DIR / "similar_songs"

TRAINING_DATA_DIR = Path("data/nj_training_data")
TRAINING_DATA_BIRDS_DIR = TRAINING_DATA_DIR / "birds"
TRAINING_DATA_DIR_SONGS = TRAINING_DATA_DIR / "songs"

OUTPUT_DIR = Path("data/nj_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Audio and RAVE settings
# ============================================================
SR = 44100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# RAVE was trained on 20-second mono snippets.
ENCODE_CHUNK_SECONDS = 10.0
ENCODE_HOP_SECONDS = 10.0
ENCODE_CHUNK_SAMPLES = int(ENCODE_CHUNK_SECONDS * SR)
ENCODE_HOP_SAMPLES = int(ENCODE_HOP_SECONDS * SR)

# The prepared training snippets are normalized to RMS 0.1.
TARGET_RMS = 0.10

# ============================================================
# Generation settings
# ============================================================
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Number of generated latent points between two neighboring anchors.
NUM_INTERMEDIATE_POINTS = 4

# With 20s chunks and 4 intermediate points:
#   anchor hold uses roughly 0-4s,
#   intermediate 1 uses 4-8s,
#   intermediate 2 uses 8-12s,
#   intermediate 3 uses 12-16s,
#   intermediate 4 uses 16-20s.
INTERMEDIATE_SLICE_SECONDS = ENCODE_CHUNK_SECONDS / (NUM_INTERMEDIATE_POINTS + 1)
ANCHOR_HOLD_SECONDS = INTERMEDIATE_SLICE_SECONDS

CROSSFADE_SECONDS = 0.35
CROSSFADE_SAMPLES = int(CROSSFADE_SECONDS * SR)

# t range for intermediate points. Values away from 0 and 1 avoid simply decoding the anchors.
T_MIN = 0.05
T_MAX = 0.95

CURVE_AMOUNT = 0.0 # 0.0 means linear interpolation, higher values mean more curved
JITTER_AMOUNT = 0.0 #0.03 means up to 3% of the distance between two points

AUDIO_EXTS = {".wav", ".mp3"}

print("Device:", DEVICE)
print("Intermediate slice seconds:", INTERMEDIATE_SLICE_SECONDS)


## 2. Load the RAVE model


In [ ]:
if not RAVE_MODEL_PATH.exists():
    raise FileNotFoundError(f"RAVE model not found: {RAVE_MODEL_PATH}")

rave = torch.jit.load(str(RAVE_MODEL_PATH), map_location=DEVICE)
rave.eval()
print("Loaded RAVE model:")
print(rave)


## 3. Load audio files

Labels used below:

- `nj`: the full Nightjar song, chunked (`ENCODE_CHUNK_SECONDS` defines the length of the snippets)
- `bird_generation`: bird audio, chunked (`ENCODE_CHUNK_SECONDS` defines the length of the snippets)
- `bird_training`: prepared bird/nature training snippets (20 seconds max)
- `song_training_nightjar`: prepared Nightjar training snippets (20 seconds max)
- `song_training_similar`: prepared similar-song training snippets (20 seconds max)

We seperate between `nj` and `song_training_nightjar`, since the `nj` snippets may be chosen to be shorter/longer than the 20-second snippets used for training. Therefore, it is expected that `song_training_nightjar` and `nj` may not overlap in latent space, even though its the same song.

In [ ]:
def find_audio_files():
    files, labels = [], []

    if NIGHTJAR_SONG.exists():
        files.append(NIGHTJAR_SONG)
        labels.append("nj")
    else:
        raise FileNotFoundError(f"Nightjar song not found: {NIGHTJAR_SONG}")
    
    if BIRDS_FOR_GENERATION_DIR.exists():
        for p in sorted(BIRDS_FOR_GENERATION_DIR.rglob("*")):
            if p.suffix.lower() in AUDIO_EXTS:
                files.append(p)
                labels.append("bird_generation")
    
    if SIMILAR_SONGS_FOR_GENERATION_DIR.exists():
        for p in sorted(SIMILAR_SONGS_FOR_GENERATION_DIR.rglob("*")):
            if p.suffix.lower() in AUDIO_EXTS:
                files.append(p)
                labels.append("similar_songs_generation")

    if TRAINING_DATA_BIRDS_DIR.exists():
        for p in sorted(TRAINING_DATA_BIRDS_DIR.rglob("*")):
            if p.suffix.lower() in AUDIO_EXTS:
                files.append(p)
                labels.append("bird_training")

    if TRAINING_DATA_DIR_SONGS.exists():
        for p in sorted(TRAINING_DATA_DIR_SONGS.rglob("*")):
            if p.suffix.lower() not in AUDIO_EXTS:
                continue
            files.append(p)
            if "nightjar" in p.stem.lower():
                labels.append("song_training_nightjar")
            else:
                labels.append("song_training_similar")

    return files, labels


audio_files, file_labels = find_audio_files()
print("Files found:", len(audio_files))
print(pd.Series(file_labels).value_counts())


In [ ]:
def load_audio(path):
    # Load mono audio at the RAVE training sample rate.
    y, _ = librosa.load(path, sr=SR, mono=True)
    y = np.nan_to_num(y).astype(np.float32)
    peak = np.max(np.abs(y)) + 1e-9
    if peak > 1.0:
        y = y / peak
    return y


audio_cache = {}
for path in tqdm(audio_files, desc="Loading audio"):
    audio_cache[str(path)] = load_audio(path)

print("Loaded audio files:", len(audio_cache))


## 4. Helper functions


In [ ]:
def normalize_audio(y, target_rms=TARGET_RMS):
    # Normalize to target RMS and prevent clipping.
    y = np.asarray(y, dtype=np.float32)
    y = np.nan_to_num(y)
    y = y - np.mean(y)
    rms = np.sqrt(np.mean(y**2)) + 1e-12
    if rms > 1e-10:
        y = y * (target_rms / rms)
    peak = np.max(np.abs(y)) + 1e-12
    if peak > 0.98:
        y = 0.98 * y / peak
    return y.astype(np.float32)


def normalize_for_rave_encoding(y, target_rms=TARGET_RMS):
    # Use the same loudness normalization before encoding all chunks.
    return normalize_audio(y, target_rms=target_rms)


def show_audio(title, y, normalize=True):
    display(Markdown(f"### {title}"))
    y = np.asarray(y, dtype=np.float32)
    if normalize:
        y = normalize_audio(y)
    print("duration:", len(y) / SR, "rms:", np.sqrt(np.mean(y**2)), "peak:", np.max(np.abs(y)))
    display(Audio(y, rate=SR))


def to_tensor(y):
    y = np.asarray(y, dtype=np.float32)
    return torch.from_numpy(y).float().view(1, 1, -1).to(DEVICE)


def from_tensor(x):
    x = x.detach().cpu().numpy()
    return np.nan_to_num(np.squeeze(x)).astype(np.float32)


@torch.no_grad()
def rave_encode(y):
    x = to_tensor(y)
    if not hasattr(rave, "encode"):
        raise RuntimeError("This RAVE TorchScript model has no encode method.")
    z = rave.encode(x)
    if isinstance(z, (tuple, list)):
        z = z[0]
    z = from_tensor(z)
    if z.ndim == 1:
        z = z[None, :]
    elif z.ndim == 3:
        z = z[0]
    return z.astype(np.float32)


@torch.no_grad()
def rave_decode(z_np):
    z_np = np.asarray(z_np, dtype=np.float32)
    if z_np.ndim == 2:
        z_t = torch.from_numpy(z_np[None, :, :]).float().to(DEVICE)
    elif z_np.ndim == 3:
        z_t = torch.from_numpy(z_np).float().to(DEVICE)
    else:
        raise ValueError(f"Unexpected latent shape: {z_np.shape}")
    if not hasattr(rave, "decode"):
        raise RuntimeError("This RAVE TorchScript model has no decode method.")
    y = rave.decode(z_t)
    if isinstance(y, (tuple, list)):
        y = y[0]
    return from_tensor(y).astype(np.float32)


def equal_power_crossfade(a, b, fade_samples=CROSSFADE_SAMPLES):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    fade_samples = min(fade_samples, len(a), len(b))
    if fade_samples <= 0:
        return np.concatenate([a, b]).astype(np.float32)
    t = np.linspace(0, 1, fade_samples)
    fade_out = np.cos(t * np.pi / 2)
    fade_in = np.sin(t * np.pi / 2)
    overlap = a[-fade_samples:] * fade_out + b[:fade_samples] * fade_in
    return np.concatenate([a[:-fade_samples], overlap, b[fade_samples:]]).astype(np.float32)


def append_crossfade(current, segment):
    if current is None:
        return segment.astype(np.float32)
    return equal_power_crossfade(current, segment)


def chunk_starts(y):
    if len(y) < ENCODE_CHUNK_SAMPLES:
        return []
    return list(range(0, len(y) - ENCODE_CHUNK_SAMPLES + 1, ENCODE_HOP_SAMPLES))


def audio_features(y):
    y = np.asarray(y, dtype=np.float32)
    rms = float(np.sqrt(np.mean(y**2)) + 1e-12)
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=SR)[0]))
    onset_env = librosa.onset.onset_strength(y=y, sr=SR)
    onset_density = float(np.mean(onset_env))
    return rms, centroid, onset_density


def latent_summary(z):
    # Fixed-length feature vector for PCA and nearest-neighbor analysis.
    z = np.asarray(z, dtype=np.float32)
    return np.concatenate([
        z.mean(axis=1),
        z.std(axis=1),
        np.percentile(z, 10, axis=1),
        np.percentile(z, 90, axis=1),
    ]).astype(np.float32)


## 5. Quick RAVE reconstruction test

- Check if audio is loaded correctly
- Encode and then decode files --> how does the reconstruction sound?

In [ ]:
for label in ["nj", "similar_songs_generation", "bird_generation", "bird_training", "song_training_nightjar", "song_training_similar"]:
    candidate_indices = [i for i, lab in enumerate(file_labels) if lab == label]
    if not candidate_indices:
        continue

    print()
    print(f"=== Example for label: {label} ===")
    test_i = random.choice(candidate_indices)
    test_file = audio_files[test_i]
    if label == "nj" or label == "bird_generation":
        test_audio = audio_cache[str(test_file)][:ENCODE_CHUNK_SAMPLES]
    else:
        test_audio = audio_cache[str(test_file)]
    test_audio_encoded = normalize_for_rave_encoding(test_audio)
    z_test = rave_encode(test_audio_encoded)
    recon_test = rave_decode(z_test)

    print("Selected file:", test_file)
    print("z_test shape:", z_test.shape)
    show_audio("Original test audio", test_audio)
    show_audio("RAVE reconstruction", recon_test)

The reconstruction is noisy but overall it works!

## 6. Encode all audio into RAVE latent space

We encode all data to see how the latent space looks like.

In [ ]:
z_list = []
summary_list = []
rows = []

PREPARED_20S_LABELS = {"song_training_similar", "song_training_nightjar", "bird_training"} 
# leave training snippets as they are -> for visualization only; generation snippets will be trimmed to ENCODE_CHUNK_SAMPLES seconds

for p, label in tqdm(list(zip(audio_files, file_labels)), desc="Encoding audio with RAVE"):
    path_str = str(p)
    y = audio_cache[path_str]

    if label in PREPARED_20S_LABELS: 
        chunks = [(0, y.astype(np.float32))]
    else:
        chunks = []
        for chunk_i, s in enumerate(chunk_starts(y)):
            chunk = y[s:s + ENCODE_CHUNK_SAMPLES]
            if len(chunk) == ENCODE_CHUNK_SAMPLES:
                chunks.append((chunk_i, chunk.astype(np.float32)))

    for chunk_i, chunk_raw in tqdm(chunks, leave=False, desc=p.name):
        chunk_encoded = normalize_for_rave_encoding(chunk_raw)
        try:
            z = rave_encode(chunk_encoded)
        except Exception as e:
            print("Encode failed:", p, chunk_i, repr(e))
            continue

        raw_rms, raw_centroid, raw_onset = audio_features(chunk_raw)
        enc_rms, enc_centroid, enc_onset = audio_features(chunk_encoded)

        z_list.append(z)
        summary_list.append(latent_summary(z))
        rows.append({
            "file": path_str,
            "source_type": label,
            "chunk_index_in_file": int(chunk_i),
            "raw_rms": raw_rms,
            "raw_spectral_centroid": raw_centroid,
            "raw_onset_density": raw_onset,
            "rms": enc_rms,
            "spectral_centroid": enc_centroid,
            "onset_density": enc_onset,
            "z_shape": str(tuple(z.shape)),
            "encoded_audio_seconds": len(chunk_encoded) / SR,
        })

metadata = pd.DataFrame(rows)
if len(summary_list) == 0:
    raise RuntimeError("No chunks were encoded. Check labels, paths, and audio lengths.")

summary_shapes = pd.Series([s.shape for s in summary_list]).value_counts()
print("Summary shapes:")
print(summary_shapes)
if len(summary_shapes) > 1:
    raise RuntimeError("Latent summaries have different shapes. Check chunk lengths.")

Z_summary = np.stack(summary_list).astype(np.float32)
print("Encoded chunks:", len(metadata))
print(metadata["source_type"].value_counts())
print("Z_summary shape:", Z_summary.shape)

metadata.to_csv(OUTPUT_DIR / "rave_latent_metadata.csv", index=False)
np.save(OUTPUT_DIR / "rave_latent_summary.npy", Z_summary)
np.savez_compressed(OUTPUT_DIR / "rave_latents_compressed.npz", **{f"z_{i:05d}": z for i, z in enumerate(z_list)})


## 7. PCA latent-space visualization

The latent space has dimension (?), therefore a PCA is applied.

In [ ]:
scaler = StandardScaler()
Z_scaled = scaler.fit_transform(Z_summary)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
Z_2d = pca.fit_transform(Z_scaled)
metadata["pca_1"] = Z_2d[:, 0]
metadata["pca_2"] = Z_2d[:, 1]
metadata.to_csv(OUTPUT_DIR / "rave_latent_metadata_with_pca.csv", index=False)

plot_config = [
    ("bird_training", 0.15, 18, "Bird Training Snippets"),
    ("bird_generation", 0.10, 35, f"Birds for Generation ({ENCODE_CHUNK_SECONDS}s chunk)"),
    ("song_training_similar", 0.20, 17, "Similar Songs Training Snippets"),
    ("song_training_nightjar", 0.50, 18, "Nightjar Training Snippets"),
    ("nj", 0.60, 35,f"Nightjar ({ENCODE_CHUNK_SECONDS}s chunk)"),
]

for zoom, fname in [(False, "rave_latent_pca_by_label.png"), (True, "rave_latent_pca_by_label_zoomed.png")]:
    plt.figure(figsize=(10, 7))
    for src, alpha, size, display_label in plot_config:
        mask = metadata["source_type"].values == src
        if mask.sum():
            plt.scatter(metadata.loc[mask, "pca_1"], metadata.loc[mask, "pca_2"], s=size, alpha=alpha, label=f"{display_label} ({mask.sum()})")
    plt.title("RAVE latent PCA by source label" + (" - zoomed in" if zoom else ""))
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    if zoom:
        plt.xlim(-8, 8)
        plt.ylim(-8, 8)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / fname, dpi=180)
    plt.show()


There are a few outliers, especially bird training data, but overall the distributions is centered at (0,0). There is a clustering of birds more on the left, songs more on the right side.

## 8. Alternative PCA component views


In [ ]:
pca_multi = PCA(n_components=6, random_state=RANDOM_SEED)
Z_pca = pca_multi.fit_transform(Z_scaled)
for i in range(Z_pca.shape[1]):
    metadata[f"pca_{i+1}"] = Z_pca[:, i]
metadata.to_csv(OUTPUT_DIR / "rave_latent_metadata_with_pca_6components.csv", index=False)

print("Explained variance ratio:")
for i, v in enumerate(pca_multi.explained_variance_ratio_, start=1):
    print(f"PC{i}: {v:.3f} ({v * 100:.1f}%)")


def plot_pca_components(pc_x=1, pc_y=2, zoom=False, xlim=(-8, 8), ylim=(-8, 8), save_name=None):
    x = Z_pca[:, pc_x - 1]
    y = Z_pca[:, pc_y - 1]
    plt.figure(figsize=(10, 7))
    for src, alpha, size, display_label in plot_config:
        mask = metadata["source_type"].values == src
        if mask.sum():
            plt.scatter(x[mask], y[mask], s=size, alpha=alpha, label=f"{display_label} ({mask.sum()})")
    plt.title(f"RAVE latent PCA: PC{pc_x} vs PC{pc_y}" + (" - zoomed" if zoom else ""))
    plt.xlabel(f"PC{pc_x} ({pca_multi.explained_variance_ratio_[pc_x - 1] * 100:.1f}% var)")
    plt.ylabel(f"PC{pc_y} ({pca_multi.explained_variance_ratio_[pc_y - 1] * 100:.1f}% var)")
    if zoom:
        plt.xlim(*xlim)
        plt.ylim(*ylim)
    plt.legend()
    plt.tight_layout()
    if save_name is not None:
        plt.savefig(OUTPUT_DIR / save_name, dpi=180)
    plt.show()

plot_pca_components(1, 2, zoom=True, save_name="pca_pc1_pc2_zoomed.png")
plot_pca_components(1, 3, zoom=True, save_name="pca_pc1_pc3_zoomed.png")
plot_pca_components(2, 3, zoom=True, save_name="pca_pc2_pc3_zoomed.png")


## 9. Score Nightjar snippets by mood / intensity

Next step: Choose suitable anchors, i. e. given points in latent space, between which we want to interpolate. One idea is to give a score to each snippet (from `nj_music_generation_data`). The goal is to build a tension/mood arc and not a complete random order in latent points.

Instead, the mood rating is based on two simple features from `metadata`:

- **onset_density**: how busy or active the snippet feels. More onsets usually mean more movement, rhythm, or intensity.
- **spectral_centroid**: how bright or sharp the snippet sounds. Higher values usually feel brighter, thinner, or more present.

**RMS** is not used since all files were normalized.

Both values are first normalized between 0 and 1 so they can be compared fairly. A low mood value means the snippet is calmer, darker, or more sparse. A high mood value means the snippet is busier, brighter, or more intense.

The final order follows a simple arc: **low → high → low**. This creates a sequence that starts calm, builds toward more active material, and then returns to a calmer feeling.

In [ ]:
nj_pool = np.where(metadata["source_type"].values == "nj")[0]
if len(nj_pool) == 0:
    raise RuntimeError("No Nightjar chunks found.")

nj_mood_df = metadata.loc[nj_pool].copy()

# Since all snippets are normalized to the same volume,
# skip RMS and use simple "feel" features instead.
for col in ["spectral_centroid", "onset_density"]:
    x = nj_mood_df[col].values.astype(float)
    nj_mood_df[col + "_norm"] = (x - x.min()) / (x.max() - x.min() + 1e-12)

# Simple mood:
# low = calm / soft / sparse
# high = bright / active / busy
nj_mood_df["mood"] = (
    0.6 * nj_mood_df["onset_density_norm"] +
    0.4 * nj_mood_df["spectral_centroid_norm"]
)

score_cols = [
    "source_type",
    "file",
    "chunk_index_in_file",
    "spectral_centroid",
    "onset_density",
    "mood",
    "pca_1",
    "pca_2",
]

display(Markdown("### Most active / bright Nightjar snippets"))
display(nj_mood_df.sort_values("mood", ascending=False)[score_cols])

# Make a non-symmetric low → high → low ordering.
# Each snippet is used only once.
# The first low section and the final low section contain different snippets.
ordered = nj_mood_df.sort_values("mood").copy()

n = len(ordered)
third = max(1, n // 3)

low = ordered.iloc[:third].copy()
mid = ordered.iloc[third:2 * third].copy()
high = ordered.iloc[2 * third:].copy()

# Split the low material into two different calm groups:
# one for the beginning, one for the ending.
low_start = low.iloc[::2].copy()
low_end = low.iloc[1::2].copy()

# The arc:
# calm snippets → medium snippets → high snippets → different calm snippets
mood_arc_df = pd.concat(
    [
        low_start.sort_values("mood"),
        mid.sort_values("mood"),
        high.sort_values("mood"),
        low_end.sort_values("mood", ascending=False),
    ],
    ignore_index=False,
)

display(Markdown("### Suggested Nightjar order: calm → active → different calm"))
display(mood_arc_df[score_cols])

# Plot the suggested non-symmetric arc
plt.figure(figsize=(13, 4))
plt.plot(range(len(mood_arc_df)), mood_arc_df["mood"], linewidth=2.5)
plt.xlabel("Position in suggested order")
plt.ylabel("Mood: calm → active")
plt.title("Suggested Nightjar mood arc: low → high → different low")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nightjar_simple_mood_arc_non_symmetric.png", dpi=180)
plt.show()

print("Suggested Order:", mood_arc_df["chunk_index_in_file"].values)

## 10. Choose music anchors and bird anchors

Edit `MUSIC_ANCHORS` manually after inspecting the mood plots. If `USE_MANUAL_MUSIC_ANCHORS = False`, the notebook creates an automatic low-to-high-to-low mood arc.

For bird anchors:

- `N_BIRD_ANCHORS` controls how many birds appear at the beginning and end.
- Leave `INTRO_BIRD_FILES` and `OUTRO_BIRD_FILES` empty for random selection.
- Put file-name substrings in those lists to force specific bird files.


In [ ]:
USE_MANUAL_MUSIC_ANCHORS = False
MUSIC_ANCHORS = [
    # Example: 4, 5, 7, 8, 9, 11, 10, 3, 6, 2, 1, 0, 13, 12
    4, 5, 7, 8, 9, 11, 10, 3, 6, 2, 1, 0, 13, 12
]
AUTO_NUM_MUSIC_ANCHORS = min(12, len(nj_mood_df))

In [ ]:
N_BIRD_ANCHORS = 4
BIRD_ANCHOR_LABEL = "bird_generation"

USE_SIMILAR_SONG_ANCHORS = True
SIMILAR_SONG_ANCHOR_LABEL = "similar_songs_generation"

SIMILAR_SONG_ANCHOR_FILES = [
    "data/nj_music_generation_data/similar_songs/Bittern [BTpSGwfeUCs]_off5s_part2.wav",
    "data/nj_music_generation_data/similar_songs/Skylark [h8eCnWe4JRE]_off10s_part2.wav",
    "data/nj_music_generation_data/similar_songs/Nightingale, Pt. 1 [D1ivfm__NaQ]_off0s_part7",
    "data/nj_music_generation_data/similar_songs/Marsh Warbler [fQ3xzydeiCI]_off5s_part2",
]

# One insert index per similar song.
# Each value means: insert the corresponding similar song after this many Nightjar anchors.
#
# Example:
#   [1, 3, 6, 9]
#
# gives roughly:
#   Nightjar
#   Similar song 1
#   Nightjar
#   Nightjar
#   Similar song 2
#   Nightjar
#   Nightjar
#   Nightjar
#   Similar song 3
#   ...
#
# To keep every similar song between two Nightjar anchors,
# every value must be from 1 to len(original_music_anchors) - 1.
SIMILAR_SONG_INSERT_INDICES = [1, 3, 6, 9]

INTRO_BIRD_FILES = [
    "data/nj_music_generation_data/birds/Andean Guan_sound_1.wav",
    "data/nj_music_generation_data/birds/Bartletts Tinamou_sound_0.wav",
    "data/nj_music_generation_data/birds/Little Tinamou_sound_1.wav",
    "data/nj_music_generation_data/birds/North Island Brown Kiwi_sound_0.wav",
]

OUTRO_BIRD_FILES = [
    "data/nj_music_generation_data/birds/West Mexican Chachalaca_sound_0.wav",
    "data/nj_music_generation_data/birds/Little Tinamou_sound_1.wav",
    "data/nj_music_generation_data/birds/Andean Guan_sound_1.wav",
    "data/nj_music_generation_data/birds/Bartletts Tinamou_sound_0.wav",
]


bird_pool = np.where(metadata["source_type"].values == BIRD_ANCHOR_LABEL)[0]
if len(bird_pool) == 0:
    raise RuntimeError(f"No bird anchors found with label {BIRD_ANCHOR_LABEL!r}.")


def choose_bird_anchors(file_patterns, fallback_n, exclude_indices=None):
    exclude_indices = set(exclude_indices or [])

    if file_patterns:
        chosen = []

        for pattern in file_patterns:
            mask = (
                (metadata["source_type"] == BIRD_ANCHOR_LABEL)
                & metadata["file"].str.contains(pattern, case=False, regex=False)
            )
            matches = metadata[mask]

            if len(matches) == 0:
                raise RuntimeError(f"No bird file matched pattern: {pattern}")

            chosen.append(matches.index[0])

        return chosen

    available = [idx for idx in bird_pool if idx not in exclude_indices]

    if len(available) < fallback_n:
        raise RuntimeError(
            f"Need {fallback_n} bird anchors, but only {len(available)} are available."
        )

    return rng.choice(available, size=fallback_n, replace=False).tolist()


def choose_similar_song_anchors(file_patterns):
    chosen = []

    for pattern in file_patterns:
        mask = (
            (metadata["source_type"] == SIMILAR_SONG_ANCHOR_LABEL)
            & metadata["file"].str.contains(pattern, case=False, regex=False)
        )
        matches = metadata[mask]

        if len(matches) == 0:
            raise RuntimeError(f"No similar-song file matched pattern: {pattern}")

        chosen.append(matches.index[0])

    return chosen


def insert_similar_song_anchors(
    original_music_anchors,
    similar_song_anchors,
    insert_indices,
):
    if len(similar_song_anchors) != len(insert_indices):
        raise ValueError(
            "SIMILAR_SONG_ANCHOR_FILES and SIMILAR_SONG_INSERT_INDICES must have "
            "the same length."
        )

    if len(original_music_anchors) < 2:
        raise ValueError(
            "Need at least two Nightjar anchors to insert similar songs between them."
        )

    for insert_idx in insert_indices:
        if insert_idx <= 0 or insert_idx >= len(original_music_anchors):
            raise ValueError(
                "Each SIMILAR_SONG_INSERT_INDICES value must place a similar song "
                f"between two Nightjar anchors. Use values from 1 to "
                f"{len(original_music_anchors) - 1}."
            )

    inserts_by_position = {}

    for insert_idx, similar_idx in zip(insert_indices, similar_song_anchors):
        inserts_by_position.setdefault(insert_idx, []).append(similar_idx)

    music_anchors_with_similar_songs = []

    for i, music_idx in enumerate(original_music_anchors):
        music_anchors_with_similar_songs.append(music_idx)

        n_music_anchors_so_far = i + 1

        if n_music_anchors_so_far in inserts_by_position:
            music_anchors_with_similar_songs.extend(
                inserts_by_position[n_music_anchors_so_far]
            )

    return music_anchors_with_similar_songs


intro_birds = choose_bird_anchors(
    INTRO_BIRD_FILES,
    fallback_n=N_BIRD_ANCHORS,
)

outro_birds = choose_bird_anchors(
    OUTRO_BIRD_FILES,
    fallback_n=N_BIRD_ANCHORS,
    exclude_indices=intro_birds,
)

# Use your new mood-arc order instead of MUSIC_ANCHORS.
# These are the actual metadata indices, not just chunk_index_in_file values.
original_music_anchors = mood_arc_df.index.tolist()
music_anchors = original_music_anchors.copy()

similar_song_anchors = []

if USE_SIMILAR_SONG_ANCHORS:
    similar_song_anchors = choose_similar_song_anchors(SIMILAR_SONG_ANCHOR_FILES)

    music_anchors = insert_similar_song_anchors(
        original_music_anchors=original_music_anchors,
        similar_song_anchors=similar_song_anchors,
        insert_indices=SIMILAR_SONG_INSERT_INDICES,
    )

main_chain = intro_birds + music_anchors + outro_birds
chosen_song_indices = main_chain

anchor_df = metadata.loc[main_chain].copy()

# Add mood columns for Nightjar rows.
# Bird and similar-song rows will get NaN here, which is fine.
mood_cols = [
    "spectral_centroid_norm",
    "onset_density_norm",
    "mood",
]

anchor_df = anchor_df.join(
    nj_mood_df[mood_cols],
    how="left",
)

anchor_df["main_order"] = np.arange(len(anchor_df))
anchor_df["is_intro_bird_anchor"] = anchor_df.index.isin(intro_birds)
anchor_df["is_outro_bird_anchor"] = anchor_df.index.isin(outro_birds)

# original_music_anchors is the pure Nightjar list.
# music_anchors is the middle chain after inserting similar-song anchors.
anchor_df["is_music_anchor"] = anchor_df.index.isin(original_music_anchors)
anchor_df["is_similar_song_anchor"] = anchor_df.index.isin(similar_song_anchors)

anchor_df.to_csv(
    OUTPUT_DIR / "chosen_mood_order_music_with_bird_and_similar_song_anchors.csv",
    index=True,
)

print("Intro birds:", intro_birds)
print("Original Nightjar music anchors:", original_music_anchors)
print("Middle chain with similar songs inserted:", music_anchors)
print("Similar song anchors:", similar_song_anchors)
print("Similar song files:")
print(metadata.loc[similar_song_anchors, "file"].tolist())
print("Similar song insert indices:", SIMILAR_SONG_INSERT_INDICES)
print("Outro birds:", outro_birds)
print("Total chain length:", len(main_chain))
print("Anchor source types:", metadata.loc[main_chain, "source_type"].unique())

display_cols = [
    "source_type",
    "file",
    "chunk_index_in_file",
    "rms",
    "spectral_centroid",
    "onset_density",
    "spectral_centroid_norm",
    "onset_density_norm",
    "mood",
    "pca_1",
    "pca_2",
    "main_order",
    "is_intro_bird_anchor",
    "is_music_anchor",
    "is_similar_song_anchor",
    "is_outro_bird_anchor",
]

display(anchor_df[display_cols])


In [ ]:
plt.figure(figsize=(10, 7))
for src, alpha, size, display_label in plot_config:
    if src == "song_training_nightjar":
        continue
    else:
        mask = metadata["source_type"].values == src
        if mask.sum():
            plt.scatter(metadata.loc[mask, "pca_1"], metadata.loc[mask, "pca_2"], s=size, alpha=alpha, label=f"{display_label} ({mask.sum()})")

plt.plot(metadata.loc[main_chain, "pca_1"], metadata.loc[main_chain, "pca_2"], linewidth=2, alpha=0.65, label="Final anchor chain")
plt.scatter(metadata.loc[music_anchors, "pca_1"], metadata.loc[music_anchors, "pca_2"], s=60, marker="X", alpha=0.95, label=f"Music anchors ({len(music_anchors)})")
plt.scatter(metadata.loc[intro_birds, "pca_1"], metadata.loc[intro_birds, "pca_2"], s=60, marker="*", alpha=0.95, label=f"Intro bird anchors ({len(intro_birds)})")
plt.scatter(metadata.loc[outro_birds, "pca_1"], metadata.loc[outro_birds, "pca_2"], s=60, marker="*", alpha=0.95, label=f"Outro bird anchors ({len(outro_birds)})")
plt.title("Music anchors with bird intro/outro anchors")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.xlim(-2, 2)
plt.ylim(-2, 2)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "manual_music_birds_anchor_chain_pca.png", dpi=180)
plt.show()


## 11. Preview selected anchors


In [ ]:
def get_anchor_audio(idx):
    row = metadata.loc[idx]
    y = audio_cache[row["file"]]
    if row["source_type"] == "nj":
        start = int(row["chunk_index_in_file"] * ENCODE_HOP_SECONDS * SR)
        end = start + ENCODE_CHUNK_SAMPLES
        return y[start:end].astype(np.float32)
    return y[:ENCODE_CHUNK_SAMPLES].astype(np.float32)


display(Markdown("## Intro bird anchors"))
for idx in intro_birds:
    show_audio(f"Intro bird original {idx}", get_anchor_audio(idx))
    show_audio(f"Intro bird RAVE reconstruction {idx}", rave_decode(z_list[idx]))

display(Markdown("## First few music anchors"))
for idx in music_anchors[:5]:
    show_audio(f"Music anchor original {idx}", get_anchor_audio(idx))
    show_audio(f"Music anchor RAVE reconstruction {idx}", rave_decode(z_list[idx]))

display(Markdown("## Outro bird anchors"))
for idx in outro_birds:
    show_audio(f"Outro bird original {idx}", get_anchor_audio(idx))
    show_audio(f"Outro bird RAVE reconstruction {idx}", rave_decode(z_list[idx]))


## 12. Render with fixed intermediate latent points

For every neighboring anchor pair, the notebook creates exactly `NUM_INTERMEDIATE_POINTS` generated latent points. Each generated point is decoded and a successive slice of the decoded 20-second audio is kept.


In [ ]:
def decode_latent_slice(z, slice_index, slice_seconds=INTERMEDIATE_SLICE_SECONDS):
    y = rave_decode(z)
    target = int(slice_seconds * SR)
    start = int(slice_index * slice_seconds * SR)
    end = start + target
    if len(y) < end:
        y = np.pad(y, (0, end - len(y)))
    seg = y[start:end]
    if len(seg) < target:
        seg = np.pad(seg, (0, target - len(seg)))
    return normalize_audio(seg)


def render_anchor_hold_fixed(idx, section_name, slice_index=0):
    z = z_list[idx]
    seg = decode_latent_slice(z, slice_index=slice_index, slice_seconds=ANCHOR_HOLD_SECONDS)
    z_pca = pca.transform(scaler.transform(latent_summary(z).reshape(1, -1)))[0]
    rows = [{
        "section": section_name,
        "local_step": 0,
        "t": 0.0,
        "pca_1": float(z_pca[0]),
        "pca_2": float(z_pca[1]),
        "anchor_from": int(idx),
        "anchor_to": int(idx),
        "decoded_segment_rms": float(np.sqrt(np.mean(seg**2))),
        "event_type": "anchor_hold",
        "slice_index": int(slice_index),
    }]
    return normalize_audio(seg), rows


def render_between_fixed_intermediates(idx_a, idx_b, section_name, seed_offset=0, num_intermediate_points=NUM_INTERMEDIATE_POINTS):
    z0 = z_list[idx_a]
    z1 = z_list[idx_b]
    min_t = min(z0.shape[-1], z1.shape[-1])
    z0 = z0[..., :min_t]
    z1 = z1[..., :min_t]

    ts = np.linspace(T_MIN, T_MAX, num_intermediate_points)
    rng_local = np.random.default_rng(RANDOM_SEED + seed_offset)
    direction = z1 - z0
    direction_scale = np.sqrt(np.mean(direction**2)) + 1e-12

    rand = rng_local.normal(size=z0.shape).astype(np.float32)
    rand = rand / (np.sqrt(np.mean(rand**2)) + 1e-12)
    mid = 0.5 * (z0 + z1) + CURVE_AMOUNT * direction_scale * rand

    y = None
    rows = []
    for i, t in enumerate(ts):
        z_base = ((1 - t)**2) * z0 + 2 * (1 - t) * t * mid + (t**2) * z1
        noise = rng_local.normal(size=z_base.shape).astype(np.float32)
        noise = noise / (np.sqrt(np.mean(noise**2)) + 1e-12)
        z_gen = z_base + noise * (JITTER_AMOUNT * direction_scale)

        seg = decode_latent_slice(z_gen, slice_index=i + 1, slice_seconds=INTERMEDIATE_SLICE_SECONDS)
        y = append_crossfade(y, seg)
        z_pca = pca.transform(scaler.transform(latent_summary(z_gen).reshape(1, -1)))[0]

        rows.append({
            "section": section_name,
            "local_step": i,
            "t": float(t),
            "pca_1": float(z_pca[0]),
            "pca_2": float(z_pca[1]),
            "anchor_from": int(idx_a),
            "anchor_to": int(idx_b),
            "decoded_segment_rms": float(np.sqrt(np.mean(seg**2))),
            "event_type": "intermediate",
            "slice_index": int(i + 1),
        })
    return normalize_audio(y), rows


def render_chain_fixed_intermediates(indices, section_name="fixed_intermediate_chain", seed_base=0):
    if len(indices) < 2:
        raise ValueError("Need at least two anchors.")
    y = None
    rows = []
    for j, idx in enumerate(indices):
        hold_y, hold_rows = render_anchor_hold_fixed(idx, section_name=f"{section_name}_hold_{j}", slice_index=0)
        y = append_crossfade(y, hold_y)
        rows.extend(hold_rows)
        if j < len(indices) - 1:
            trans_y, trans_rows = render_between_fixed_intermediates(idx, indices[j + 1], section_name=f"{section_name}_intermediate_{j}", seed_offset=seed_base + j)
            y = append_crossfade(y, trans_y)
            rows.extend(trans_rows)
    return normalize_audio(y), rows


In [ ]:
song, path_rows = render_chain_fixed_intermediates(main_chain, section_name="fixed_intermediate_generation", seed_base=200)
path_df = pd.DataFrame(path_rows)

out_wav = OUTPUT_DIR / "rave_latent_fixed_intermediates.wav"
sf.write(out_wav, song, SR)
path_df.to_csv(OUTPUT_DIR / "generated_path_points_fixed_intermediates.csv", index=False)

print("Saved:", out_wav)
print("Duration:", len(song) / SR)
print("Number of anchors:", len(main_chain))
print("Number of transitions:", len(main_chain) - 1)
print("Intermediate points per transition:", NUM_INTERMEDIATE_POINTS)
print("Anchor hold seconds:", ANCHOR_HOLD_SECONDS)
print("Intermediate slice seconds:", INTERMEDIATE_SLICE_SECONDS)
print("RMS:", np.sqrt(np.mean(song**2)))
print("Peak:", np.max(np.abs(song)))

display(Markdown("# Final RAVE generated song — fixed intermediate points"))
display(Audio(song, rate=SR))


## 13. Analyze the generated song


In [ ]:
def analysis_curves(y, sr=SR):
    hop = 1024
    rms_curve = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop)[0]
    centroid_curve = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
    onset_curve = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop)
    times = librosa.frames_to_time(np.arange(len(rms_curve)), sr=sr, hop_length=hop)
    n = min(len(times), len(rms_curve), len(centroid_curve), len(onset_curve))
    return times[:n], rms_curve[:n], centroid_curve[:n], onset_curve[:n]


times, rms_curve, centroid_curve, onset_curve = analysis_curves(song)
for arr, title, ylabel, fname in [
    (rms_curve, "RMS energy over time", "RMS", "generated_rms_curve.png"),
    (centroid_curve, "Spectral centroid over time", "Hz", "generated_centroid_curve.png"),
    (onset_curve, "Onset strength over time", "Onset strength", "generated_onset_curve.png"),
]:
    plt.figure(figsize=(14, 4))
    plt.plot(times, arr)
    plt.title(title)
    plt.xlabel("Time (s)")
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / fname, dpi=180)
    plt.show()


## 14. Animated PCA path

The gray points are generated anchor-hold and intermediate points from `path_df`. The moving black dot follows the currently decoded generated point.


In [ ]:
path_anim = path_df.copy().reset_index(drop=True)

display(Markdown("## Generated audio"))
display(Audio(song, rate=SR))

fig, ax = plt.subplots(figsize=(9, 7))
X_MIN, X_MAX = -2, 2
Y_MIN, Y_MAX = -2.5, 2.5

for src, alpha, size, display_label in plot_config:
    mask = metadata["source_type"].values == src
    if mask.sum():
        ax.scatter(metadata.loc[mask, "pca_1"], metadata.loc[mask, "pca_2"], s=size, alpha=alpha, label=f"{display_label} ({mask.sum()})")

bird_anchor_indices = list(dict.fromkeys(list(intro_birds) + list(outro_birds)))
music_anchor_indices = [idx for idx in main_chain if idx not in bird_anchor_indices]

anchor_x = metadata.loc[main_chain, "pca_1"].values
anchor_y = metadata.loc[main_chain, "pca_2"].values
ax.plot(anchor_x, anchor_y, linestyle="--", linewidth=1.4, alpha=0.45, label="Anchor order")

if len(music_anchor_indices):
    ax.scatter(metadata.loc[music_anchor_indices, "pca_1"], metadata.loc[music_anchor_indices, "pca_2"], s=55, marker="X", alpha=0.85, label=f"Music anchors ({len(music_anchor_indices)})")
if len(bird_anchor_indices):
    ax.scatter(metadata.loc[bird_anchor_indices, "pca_1"], metadata.loc[bird_anchor_indices, "pca_2"], s=55, marker="*", alpha=0.95, color="darkorange", label=f"Bird anchors ({len(bird_anchor_indices)})")

ax.scatter(path_anim["pca_1"], path_anim["pca_2"], s=12, alpha=0.45, color="yellow", label=f"Generated intermediate / hold points ({len(path_anim)})")
ax.plot(path_anim["pca_1"], path_anim["pca_2"], linewidth=1.5, alpha=0.22, label="Full generated latent path")

done_line, = ax.plot([], [], linewidth=3, alpha=0.9, label="Completed generated path")
current_point, = ax.plot([], [], "o", markersize=4, color="black", label="Current generated point")
current_anchor_line, = ax.plot([], [], linewidth=4, alpha=0.55, label="Current anchor transition")

ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(Y_MIN, Y_MAX)
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
ax.set_title("Generated path through PCA space")
ax.legend(loc="best", fontsize=8)


def update(frame):
    row = path_anim.iloc[frame]
    done_line.set_data(path_anim["pca_1"].iloc[:frame + 1], path_anim["pca_2"].iloc[:frame + 1])
    current_point.set_data([row["pca_1"]], [row["pca_2"]])
    anchor_from = int(row["anchor_from"])
    anchor_to = int(row["anchor_to"])
    current_anchor_line.set_data(
        [metadata.loc[anchor_from, "pca_1"], metadata.loc[anchor_to, "pca_1"]],
        [metadata.loc[anchor_from, "pca_2"], metadata.loc[anchor_to, "pca_2"]],
    )
    ax.set_title(f"Step {frame + 1}/{len(path_anim)} | {row.get('event_type', '')} | {anchor_from} → {anchor_to}")
    return done_line, current_point, current_anchor_line


anim = FuncAnimation(fig, update, frames=len(path_anim), interval=120, blit=False)
fig.savefig(OUTPUT_DIR / "animated_pca_plot_static.png", dpi=180, bbox_inches="tight")
plt.close(fig)

display(Markdown("## Animated PCA path with anchors and intermediate points"))
display(HTML(anim.to_jshtml()))


## 15. Save animation and tuning notes


In [ ]:
html_path = OUTPUT_DIR / "animated_pca_path.html"
with open(html_path, "w", encoding="utf-8") as f:
    f.write(anim.to_jshtml())
print("Saved HTML animation:", html_path)

# Optional MP4 export if ffmpeg is installed:
# mp4_path = OUTPUT_DIR / "animated_pca_path.mp4"
# anim.save(str(mp4_path), writer="ffmpeg", fps=8, dpi=120)
# print("Saved MP4 animation:", mp4_path)


## 16. Tuning notes

If the result sounds too close to the anchors:

```python
JITTER_AMOUNT = 0.06
CURVE_AMOUNT = 0.35
T_MIN = 0.25
T_MAX = 0.75
```

If the result sounds too glitchy, metallic, or robotic:

```python
JITTER_AMOUNT = 0.01
CURVE_AMOUNT = 0.10
T_MIN = 0.10
T_MAX = 0.90
```

If transitions should contain more generated material:

```python
NUM_INTERMEDIATE_POINTS = 6
```

If the song is too long, reduce the number of anchors or reduce `NUM_INTERMEDIATE_POINTS`. If it is too short, add more anchors or increase `NUM_INTERMEDIATE_POINTS`.
